# Flood Frequency Analysis of NVE Discharge Data

This notebook performs hydrological extreme value analysis including:
- Time series visualization
- Seasonal pattern analysis
- Flood rose plot
- Annual maximum extraction
- GEV distribution fitting (L-moments)
- Diagnostic plots
- Return level estimation


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import genextreme
import lmoments3 as lm
import lmoments3.distr as lm_distr

plt.style.use('seaborn-v0_8')

In [ ]:
# ---------- Read Data ----------

file_path = '62.5.0-Vannføring-dogn-v1 (1).csv'

df = pd.read_csv(file_path, sep=';', skiprows=1)
df.columns = ['time', 'discharge', 'corrected', 'quality']

df['time'] = pd.to_datetime(df['time'], utc=True)
df['discharge'] = pd.to_numeric(df['discharge'], errors='coerce')

df = df.dropna(subset=['discharge'])
df = df.set_index('time')

df.head()

## Time Series and Annual Maxima

In [ ]:
annual_max = df['discharge'].resample('Y').max()

plt.figure(figsize=(12,6))
plt.plot(df.index, df['discharge'], alpha=0.5)
plt.scatter(annual_max.index, annual_max.values,
            color='red', label='Annual maxima')

plt.title('Daily Discharge with Annual Maxima')
plt.ylabel('Discharge (m3/s)')
plt.legend()
plt.show()

## Seasonality Analysis

In [ ]:
monthly_mean = df['discharge'].groupby(df.index.month).mean()

plt.figure(figsize=(8,5))
plt.plot(monthly_mean.index, monthly_mean.values, marker='o')
plt.xticks(range(1,13))
plt.xlabel('Month')
plt.ylabel('Mean discharge (m3/s)')
plt.title('Average Monthly Seasonality')
plt.grid()
plt.show()

## Flood Rose

In [ ]:
threshold = df['discharge'].quantile(0.95)
floods = df[df['discharge'] > threshold]

theta = 2*np.pi * floods.index.dayofyear / 365.25
r = floods['discharge']

plt.figure(figsize=(6,6))
ax = plt.subplot(111, polar=True)
ax.scatter(theta, r)
ax.set_title('Flood Rose (Q > 95th percentile)')
plt.show()

## Fit GEV Distribution (L-moments)

In [ ]:
annual_max = annual_max.dropna()

lmom = lm.lmom_ratios(annual_max.values, nmom=4)
gev_fit = lm_distr.gev.lmom_fit(lmom)

shape = gev_fit['c']
loc = gev_fit['loc']
scale = gev_fit['scale']

print('GEV Parameters')
print('Shape:', shape)
print('Location:', loc)
print('Scale:', scale)

## Diagnostic Plots

In [ ]:
sorted_data = np.sort(annual_max.values)
prob = (np.arange(1, len(sorted_data)+1)-0.5)/len(sorted_data)

model_quant = genextreme.ppf(prob, shape, loc=loc, scale=scale)

plt.figure()
plt.scatter(model_quant, sorted_data)
plt.plot(model_quant, model_quant)
plt.xlabel('Model quantiles')
plt.ylabel('Observed quantiles')
plt.title('QQ Plot')
plt.grid()
plt.show()

In [ ]:
model_cdf = genextreme.cdf(sorted_data, shape, loc=loc, scale=scale)

plt.figure()
plt.scatter(prob, model_cdf)
plt.plot([0,1],[0,1])
plt.xlabel('Empirical probability')
plt.ylabel('Model probability')
plt.title('PP Plot')
plt.grid()
plt.show()

## Return Level Plot

In [ ]:
return_periods = np.array([2,5,10,20,50,100,200])
return_levels = genextreme.ppf(1-1/return_periods,
                               shape, loc=loc, scale=scale)

plt.figure()
plt.semilogx(return_periods, return_levels, marker='o')
plt.xlabel('Return period (years)')
plt.ylabel('Return level (m3/s)')
plt.title('Return Level Plot')
plt.grid()
plt.show()

## Design Floods

In [ ]:
design_periods = [10,50,100,200]

for T in design_periods:
    qT = genextreme.ppf(1-1/T, shape, loc=loc, scale=scale)
    print(f'{T}-year flood: {qT:.2f} m3/s')